# Recipe Flowchart -- dev notebook


Mirrors `src/build_site.py` step by step, for exploring the pipeline interactively: load a recipe, extract a Gozinto-style ingredient/operation graph with one model, render it as an HTML table, then compare all four models on cost/latency/structure.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, "src")

from dotenv import load_dotenv
load_dotenv()

from recipe_parser import parse_recipe
from gozinto_render import to_table_html
from build_site import RECIPES, RUNS, _node_counts

## 1. Load a recipe

The complex example is a photo of my own handwritten notes -- bilingual (Korean/English) shorthand with hand-drawn brackets already showing which sub-steps run in parallel. Good stress test for the vision-extraction path.

In [2]:
recipe = RECIPES[2]  # the handwritten Choux au Craquelin photo
recipe["input"]

WindowsPath('C:/Users/crunc/Documents/Code/data-projects/recipe-flowchart/data/recipes/03_choux_au_craquelin.jpg')

## 2. Extract the node graph

One forced tool-use call returns a title plus a flat list of nodes: `ingredient` nodes (name, amount, always a layer-0 leaf) and `operation` nodes (a short technique label plus `inputs`, the ids of whatever it consumes) -- the field that encodes the dependency graph.

In [3]:
result = parse_recipe(recipe["input"], provider="anthropic", model="claude-sonnet-5")

print(result.title)
for node in result.nodes:
    if node["type"] == "ingredient":
        print(f"  {node['id']:>3}  {node['name']:<30}  {node['amount']}")
    else:
        print(f"  {node['id']:>3}  [{node['technique']}]  inputs={node['inputs']}  ({node['detail']})")


   i1  brown sugar                     115g
   i2  butter                          115g
   i3  AP flour                        115g
   i4  salt                            1/8 tsp
   o1  [cream]  inputs=['i1', 'i2']  (None)
   o2  [combine until crumbly]  inputs=['o1', 'i3', 'i4']  (None)
   o3  [press into rectangle]  inputs=['o2']  (6x8 rectangle)
   o4  [roll out and freeze]  inputs=['o3']  (12x14, 1/8", freeze 5 min)
   o5  [cut rounds]  inputs=['o4']  (18x 2" rounds)
   i5  water                           235g
   i6  butter                          84g
   i7  sugar                           8g
   i8  salt                            2g
   i9  AP flour                        128g
  i10  eggs                            4 large
   o6  [boil panade]  inputs=['i5', 'i6', 'i7', 'i8', 'i9']  (170F)
   o7  [cool]  inputs=['o6']  (None)
   o8  [mix in eggs]  inputs=['o7', 'i10']  (None)
   o9  [top with craquelin round and bake]  inputs=['o8', 'o5']  (400F 10-12 min)
  o10  [rest with oven 

## 3. Render the Gozinto table

`to_table_html` layers the graph (ingredients + zero-input setup steps at layer 0, each operation 1 + the deepest layer of its own inputs) and emits a plain HTML `<table>` -- ingredient rows on the left, one column per layer, `rowspan` merging the cells that feed a shared operation. No client-side library, unlike v1's Mermaid diagrams.

In [4]:
from IPython.display import HTML

HTML(to_table_html(result.nodes))

The table renders directly in the notebook output above (no external tool needed, unlike v1's Mermaid diagrams which had to be pasted into mermaid.live).

## 4. Compare models

The real question: does a $1/MTok model get the *dependency graph* right, not just the ingredient list? Run the same recipe through all four models and compare cost, latency, and structure -- ingredient count, operation count, and how many operations are real merges (2+ inputs), a proxy for whether the model caught the convergence points.

In [5]:
for provider, model, label in RUNS:
    r = parse_recipe(recipe["input"], provider=provider, model=model)
    n_ingredients, n_operations, n_merges = _node_counts(r.nodes)
    print(f"{label:<16}  {r.input_tokens:>5}in {r.output_tokens:>4}out  "
          f"${r.estimated_cost_usd:.4f}  {r.latency_s:5.1f}s  "
          f"{n_ingredients:>2} ingredients, {n_operations:>2} ops ({n_merges} merges)")

Claude Haiku 4.5   3209in  959out  $0.0080    5.9s  10 ingredients,  9 ops (5 merges)
Claude Sonnet 5    3894in 1237out  $0.0302   10.9s  10 ingredients, 10 ops (5 merges)
GPT-4o mini       37690in  529out  $0.0060    7.6s  10 ingredients,  5 ops (3 merges)
GPT-4o             1960in  609out  $0.0110    4.4s  10 ingredients,  7 ops (3 merges)
Gemini 3.5 Flash-Lite   2092in  758out  $0.0025    2.5s  10 ingredients,  5 ops (4 merges)
Gemini 3.5 Flash   2092in  908out  $0.0113   31.5s  10 ingredients,  9 ops (5 merges)


## 5. Build the full site

Runs all three recipes x all four models and regenerates `docs/index.html`.

In [6]:
import build_site
build_site.main()

  cooked | Claude Haiku 4.5       |  2608in 2114out | $0.0132 |  12.8s | 23 ingredients, 15 ops (13 merges)  [cached]
  cooked | Claude Sonnet 5        |  3174in 2403out | $0.0456 |  15.7s | 22 ingredients, 16 ops (14 merges)  [cached]
  cooked | GPT-4o mini            |  1577in 1305out | $0.0010 |  12.2s | 24 ingredients,  9 ops (6 merges)  [cached]
  cooked | GPT-4o                 |  1577in 1587out | $0.0198 |   8.6s | 22 ingredients, 17 ops (13 merges)  [cached]
  cooked | Gemini 3.5 Flash-Lite  |  1772in 1878out | $0.0052 |   4.7s | 21 ingredients, 15 ops (13 merges)  [cached]
  cooked | Gemini 3.5 Flash       |  1772in 2001out | $0.0207 |  18.8s | 22 ingredients, 14 ops (12 merges)  [cached]
   baked | Claude Haiku 4.5       |  1975in 1100out | $0.0075 |   6.2s | 11 ingredients, 10 ops (4 merges)  [cached]
   baked | Claude Sonnet 5        |  2355in 1161out | $0.0245 |   8.1s | 11 ingredients,  9 ops (4 merges)  [cached]
   baked | GPT-4o mini            |  1110in  675out | $0.00